# AIMO3 — Meta-Harness + TIR

1. Install vLLM from bundled wheels
2. Load GPT-OSS-120B
3. Run mini meta-harness search on reference problems → discover best retrieval harness
4. Use best harness + TIR + voting to solve competition problems
5. Submit

In [ ]:
# ============================================================
# Cell 0: Install deps + set environment
# ============================================================
import os, sys, subprocess, glob

# Set env vars BEFORE importing anything (top notebook pattern)
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'

# Show inputs
print("=== /kaggle/input contents ===")
for d in sorted(os.listdir('/kaggle/input')):
    full = os.path.join('/kaggle/input', d)
    print(f"  {d}/")
    if os.path.isdir(full):
        for f2 in sorted(os.listdir(full))[:5]:
            print(f"    {f2}/")

# Try to find and extract the aimo-3-utils wheels + tiktoken
# Search for tar.gz in all inputs
archives = glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
print(f"\nArchives: {archives}")

temp_dir = '/kaggle/tmp/setup'
if archives:
    if not os.path.exists(os.path.join(temp_dir, 'wheels')):
        os.makedirs(temp_dir, exist_ok=True)
        for ar in archives:
            subprocess.run(['tar', '-xzf', ar, '-C', temp_dir])
    
    # Set tiktoken path if extracted
    tiktoken_dir = os.path.join(temp_dir, 'tiktoken_encodings')
    if os.path.exists(tiktoken_dir):
        os.environ['TIKTOKEN_ENCODINGS_BASE'] = tiktoken_dir
        print(f'TIKTOKEN_ENCODINGS_BASE={tiktoken_dir}')
    
    # Install vllm from wheels
    wheels_dirs = glob.glob(f'{temp_dir}/**/wheels', recursive=True) or [temp_dir]
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        '--no-index', '--find-links', wheels_dirs[0], 'vllm', 'openai_harmony'])
    print('Installed from wheels.')
else:
    # No wheels - check if vllm is pre-installed (docker image)
    try:
        import vllm
        print(f'vLLM pre-installed: {vllm.__version__}')
    except ImportError:
        print('WARNING: vLLM not available and no wheels found')

# Set tiktoken fallback: download encodings manually if not in wheels
tiktoken_dir = os.path.join(temp_dir, 'tiktoken_encodings')
if not os.path.exists(tiktoken_dir):
    os.makedirs(tiktoken_dir, exist_ok=True)
    # The model needs these files but can't download offline
    # Check if they exist elsewhere
    for candidate in glob.glob('/kaggle/input/**/tiktoken_encodings', recursive=True):
        os.environ['TIKTOKEN_ENCODINGS_BASE'] = candidate
        print(f'Found tiktoken at: {candidate}')
        break
    else:
        # Set the env var anyway - vLLM might handle it
        os.environ['TIKTOKEN_ENCODINGS_BASE'] = tiktoken_dir
        os.environ['TIKTOKEN_CACHE_DIR'] = tiktoken_dir
        print(f'Tiktoken dir set to: {tiktoken_dir} (may be empty)')

# Find model + corpus paths
model_path = None
for root, dirs, files in os.walk('/kaggle/input/models'):
    if 'config.json' in files:
        model_path = root
        break
if not model_path:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'config.json' in files:
            model_path = root
            break

corpus_path = None
for p in glob.glob('/kaggle/input/**/*.jsonl', recursive=True):
    corpus_path = p
    break

print(f'\nModel: {model_path}')
print(f'Corpus: {corpus_path}')

In [ ]:
# ============================================================
# Cell 1: Imports & Config
# ============================================================
import os, sys, time, re, json, traceback, threading, gc, math, glob, random
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr
from collections import Counter
from pathlib import Path
import polars as pl

# === PATHS (from Cell 0 discovery) ===
MODEL_PATH = model_path or '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
CORPUS_PATH = corpus_path  # May be None if dataset didn't mount
ANSWER_MOD = 100_000

# Meta-harness search config
MH_SEARCH_SIZE = 10
MH_ITERATIONS = 3
MH_SAMPLES_PER_PROBLEM = 4

# Final inference config
N_SAMPLES = 32
MAX_TOKENS = 8192
TEMPERATURE = 0.7
CODE_TIMEOUT = 30
TIR_MAX_RETRIES = 3

print(f'Model: {MODEL_PATH}')
print(f'Corpus: {CORPUS_PATH}')

In [ ]:
# ============================================================
# Cell 2: Launch vLLM as subprocess server
# ============================================================
import subprocess, httpx
from concurrent.futures import ThreadPoolExecutor

VLLM_PORT = 8000
SERVED_NAME = 'gpt-oss'

# Pre-load model weights into OS page cache
print(f'Pre-loading weights from {MODEL_PATH}...')
t0 = time.time()
files_to_load = []
for root, _, files in os.walk(MODEL_PATH):
    for fn in files:
        fp = os.path.join(root, fn)
        if os.path.isfile(fp):
            files_to_load.append(fp)

def _read_file(path):
    with open(path, 'rb') as f:
        while f.read(1024*1024*1024): pass

with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_read_file, files_to_load))
print(f'Pre-loaded {len(files_to_load)} files in {time.time()-t0:.1f}s')

# Start vLLM server
cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_PATH,
    '--served-model-name', SERVED_NAME,
    '--tensor-parallel-size', '1',
    '--gpu-memory-utilization', '0.92',
    '--dtype', 'auto',
    '--kv-cache-dtype', 'fp8_e4m3',
    '--max-model-len', '16384',
    '--max-num-seqs', '64',
    '--host', '0.0.0.0',
    '--port', str(VLLM_PORT),
    '--enable-prefix-caching',
    '--disable-log-stats',
    '--trust-remote-code',
]

log_file = open('vllm_server.log', 'w')
server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

# Wait for server (up to 5 min)
print('Waiting for vLLM server...')
VLLM_URL = f'http://0.0.0.0:{VLLM_PORT}/v1'
for i in range(300):
    try:
        r = httpx.get(f'{VLLM_URL}/models', timeout=2)
        if r.status_code == 200:
            print(f'Server ready in {i}s: {r.json()["data"][0]["id"]}')
            break
    except:
        pass
    rc = server_proc.poll()
    if rc is not None:
        log_file.flush()
        with open('vllm_server.log') as f:
            logs = f.read()
        print(f'Server died (code {rc}). Last 2000 chars of log:')
        print(logs[-2000:])
        raise RuntimeError(f'vLLM server died with code {rc}')
    time.sleep(1)
else:
    log_file.flush()
    with open('vllm_server.log') as f:
        print(f.read()[-2000:])
    raise RuntimeError('vLLM server timeout after 300s')

def vllm_generate(prompt, n=1, temperature=0.7, max_tokens=4096, stop=None):
    """Generate via vLLM OpenAI-compatible API."""
    payload = {
        'model': SERVED_NAME,
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': temperature,
        'max_tokens': max_tokens,
        'n': n,
    }
    if stop:
        payload['stop'] = stop
    r = httpx.post(f'{VLLM_URL}/chat/completions', json=payload, timeout=300)
    r.raise_for_status()
    return [c['message']['content'] for c in r.json()['choices']]

print('vLLM server ready.')

In [ ]:
# ============================================================
# Cell 3: Load Corpus + Shared Utilities
# ============================================================

# Load corpus (if available)
corpus = []
if CORPUS_PATH and os.path.exists(CORPUS_PATH):
    with open(CORPUS_PATH) as f:
        for line in f:
            if line.strip():
                corpus.append(json.loads(line))
    print(f'Corpus: {len(corpus)} problems')
else:
    print('No corpus file found — RAG disabled, using direct prompts only')

# Answer extraction
def _parse_number(s):
    s = s.strip().replace('\\,','').replace('\\;','').replace(',','')
    s = re.sub(r'\\text\{.*?\}', '', s)
    s = re.sub(r'\\mathrm\{.*?\}', '', s)
    try: return int(s)
    except ValueError: pass
    try:
        f = float(s)
        if f == int(f) and f == f: return int(f)
    except: pass
    m = re.search(r'(-?\d+(?:\.\d+)?)', s)
    if m:
        try:
            f = float(m.group(1))
            if f == int(f): return int(f)
        except: pass
    return None

def extract_boxed(text):
    idx = text.rfind('\\boxed')
    if idx == -1: return None
    bs = text.find('{', idx)
    if bs == -1: return None
    depth, end = 0, bs
    for i in range(bs, len(text)):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0: end = i; break
    return _parse_number(text[bs+1:end].strip())

def extract_answer(text):
    r = extract_boxed(text)
    if r is not None: return r % ANSWER_MOD
    ms = re.findall(r'```output\s*(.*?)```', text, re.DOTALL)
    if ms:
        r = _parse_number(ms[-1].strip())
        if r is not None: return r % ANSWER_MOD
    for p in [r'answer\s+is\s*[:\s]*(-?\d+)', r'answer\s*[=:]\s*(-?\d+)']:
        ms = re.findall(p, text, re.IGNORECASE)
        if ms:
            r = _parse_number(ms[-1])
            if r is not None: return r % ANSWER_MOD
    ms = re.findall(r'(?<![.\d])(-?\d+)(?![.\d])', text)
    if ms:
        r = _parse_number(ms[-1])
        if r is not None: return r % ANSWER_MOD
    return None

# Code sandbox
SANDBOX_IMPORTS = '''import math, numpy as np, sympy as sp
from sympy import *
from itertools import combinations, permutations, product as iproduct
from collections import Counter, defaultdict
from fractions import Fraction
import itertools
'''

def execute_code(code, timeout=CODE_TIMEOUT):
    so_buf, se_buf = StringIO(), StringIO()
    ns = {}
    try: exec(SANDBOX_IMPORTS, ns)
    except: pass
    exc = [None]
    def _run():
        try:
            with redirect_stdout(so_buf), redirect_stderr(se_buf): exec(code, ns)
        except Exception as e: exc[0] = e
    t = threading.Thread(target=_run, daemon=True)
    t.start(); t.join(timeout=timeout)
    if t.is_alive(): return '', f'Timeout {timeout}s', False
    if exc[0]:
        return so_buf.getvalue(), ''.join(traceback.format_exception(type(exc[0]),exc[0],exc[0].__traceback__)), False
    return so_buf.getvalue(), se_buf.getvalue(), True

def extract_code_blocks(text):
    blocks = re.findall(r'```python\s*\n(.*?)```', text, re.DOTALL)
    return blocks if blocks else re.findall(r'```\s*\n(.*?)```', text, re.DOTALL)

print('Utilities ready.')

In [ ]:
# ============================================================
# Cell 4: Seed Harnesses for Meta-Harness Search
# ============================================================

SEEDS = {}

SEEDS['no_retrieval'] = '''
def run(problem):
    q = problem["question"]
    prompt = f"Solve step by step. Write Python code to verify. Put answer in \\\\boxed{{}}.\\n\\nProblem: {q}\\n\\nSolution:"
    return {"prompt": prompt, "context_tokens": len(prompt.split())}
'''

SEEDS['bm25_k3'] = '''
import math
def run(problem):
    q = problem["question"]
    corpus = problem.get("corpus", [])
    qw = set(q.lower().split())
    scored = []
    for d in corpus:
        dw = set(d["question"].lower().split())
        overlap = len(qw & dw)
        scored.append((overlap / math.sqrt(max(len(dw),1)), d))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for _, d in scored[:3]]
    ex = ""
    for e in top:
        sol = e.get("solution","")[:400]
        ex += f"Example: {e['question']}\\nSolution: {sol}\\nAnswer: {e.get('answer','N/A')}\\n\\n"
    prompt = f"Reference:\\n{ex}\\nSolve step by step with Python code. Put answer in \\\\boxed{{}}.\\n\\nProblem: {q}\\n\\nSolution:"
    return {"prompt": prompt, "context_tokens": len(prompt.split())}
'''

SEEDS['domain_routing'] = '''
import math, re
GEO = ["triangle","circle","angle","perpendicular","inscribed","tangent","polygon"]
NT = ["prime","divisible","modulo","gcd","remainder","congruent","coprime"]
COMBO = ["how many","number of ways","permutation","combinat","probability"]
SOL_MAX = {"combinatorics":800, "geometry":300, "number_theory":400, "algebra":400}

def classify(q):
    q = q.lower()
    scores = {"geometry": sum(1 for k in GEO if k in q),
              "number_theory": sum(1 for k in NT if k in q),
              "combinatorics": sum(1 for k in COMBO if k in q)}
    for d in ["combinatorics","geometry","number_theory"]:
        if scores[d] >= 2: return d
    best = max(scores, key=scores.get)
    return best if scores[best] >= 1 else "algebra"

def run(problem):
    q = problem["question"]
    corpus = problem.get("corpus", [])
    domain = classify(q)
    dc = [d for d in corpus if classify(d["question"]) == domain]
    if len(dc) < 3: dc = corpus
    qw = set(q.lower().split())
    scored = []
    for d in dc:
        dw = set(d["question"].lower().split())
        scored.append((len(qw & dw) / math.sqrt(max(len(dw),1)), d))
    scored.sort(key=lambda x: x[0], reverse=True)
    # Jaccard diversity
    selected, sel_w = [], []
    for _, d in scored:
        if len(selected) >= 3: break
        dw = set(d["question"].lower().split())
        if all(len(dw & sw)/max(len(dw|sw),1) < 0.5 for sw in sel_w):
            selected.append(d); sel_w.append(dw)
    mc = SOL_MAX.get(domain, 400)
    ex = ""
    for e in selected:
        sol = e.get("solution","")[:mc]
        ex += f"[{domain.upper()}] {e['question']}\\nSolution: {sol}\\nAnswer: {e.get('answer','N/A')}\\n\\n"
    prompt = f"Domain: {domain}\\nReference:\\n{ex}\\nSolve step by step with Python code. Put answer in \\\\boxed{{}}.\\n\\nProblem: {q}\\n\\nSolution:"
    return {"prompt": prompt, "context_tokens": len(prompt.split())}
'''

print(f'Seed harnesses: {list(SEEDS.keys())}')

In [ ]:
# ============================================================
# Cell 5: Meta-Harness Evaluator
# Uses vLLM server to score a harness on a set of problems
# ============================================================

def evaluate_harness(harness_code, problems, retrieval_corpus, n_samples=MH_SAMPLES_PER_PROBLEM):
    """Score a harness: run it on problems, solve with LLM, check answers."""
    ns = {}
    try:
        exec(harness_code, ns)
    except Exception as e:
        return {'accuracy': 0.0, 'context_cost': 0, 'error': str(e)}
    run_fn = ns.get('run')
    if not callable(run_fn):
        return {'accuracy': 0.0, 'context_cost': 0, 'error': 'no run()'}
    
    correct = 0
    total_tokens = 0
    
    for prob in problems:
        try:
            result = run_fn({'question': prob['question'], 'corpus': retrieval_corpus})
            raw_prompt = result['prompt']
            total_tokens += result.get('context_tokens', 0)
            
            # Generate via vLLM server
            responses = vllm_generate(raw_prompt, n=n_samples, temperature=0.7, max_tokens=4096)
            
            # Extract answers and vote
            answers = []
            for text in responses:
                blocks = extract_code_blocks(text)
                if blocks:
                    so, se, ok = execute_code(blocks[-1])
                    if ok and so.strip():
                        text += f'\n```output\n{so.strip()}\n```\n'
                a = extract_answer(text)
                if a is not None:
                    answers.append(a)
            
            if answers:
                predicted = Counter(answers).most_common(1)[0][0]
                expected = prob.get('answer', '')
                try:
                    if int(float(str(predicted))) == int(float(str(expected))):
                        correct += 1
                except:
                    pass
        except Exception as e:
            pass
    
    n = len(problems)
    return {
        'accuracy': correct / n if n else 0,
        'context_cost': total_tokens / n if n else 0,
        'correct': correct,
        'total': n,
    }

print('Meta-harness evaluator ready.')

In [ ]:
# ============================================================
# Cell 6: Meta-Harness Proposer (uses same vLLM server)
# ============================================================

PROPOSER_PROMPT = '''You write Python retrieval harnesses for math competition problems.
The harness must define: run(problem) -> {"prompt": str, "context_tokens": int}
problem has: "question" (str with LaTeX) and "corpus" (list of dicts with question/solution/answer).

Techniques from the Meta-Harness paper:
- Domain routing: geometry/combinatorics/number_theory/algebra
- Math-aware tokenization: preserve LaTeX tokens
- Difficulty filtering, Jaccard diversity dedup (threshold 0.5)
- Domain-adaptive solution truncation (geo:300, combo:800, nt:400, alg:400)
- Prefer corpus entries with code solutions
- Include "Put answer in \\boxed{}" in the prompt

Output ONLY valid Python code. No markdown. Must define run(problem).'''

def propose_harness(archive_text, iteration):
    """Use vLLM to propose a new harness."""
    prompt = f"{PROPOSER_PROMPT}\n\nIteration {iteration}. Prior results:\n{archive_text}\n\nWrite an improved harness:"
    responses = vllm_generate(prompt, n=1, temperature=0.8, max_tokens=4096)
    code = responses[0]
    # Clean markdown fences if present
    if '```python' in code:
        code = code.split('```python')[1].split('```')[0]
    elif '```' in code:
        code = code.split('```')[1].split('```')[0]
    return code.strip()

print('Proposer ready.')

In [ ]:
# ============================================================
# Cell 7: Run Meta-Harness Search
# ============================================================
search_start = time.time()

# Split corpus into search set + retrieval pool
rng = random.Random(42)
shuffled = list(corpus)
rng.shuffle(shuffled)
# Use problems that have answers for evaluation
with_answers = [p for p in shuffled if p.get('answer')]
search_problems = with_answers[:MH_SEARCH_SIZE]
retrieval_pool = shuffled[:500]  # retrieval corpus for harnesses

print(f'Search problems: {len(search_problems)} (with known answers)')
print(f'Retrieval pool: {len(retrieval_pool)}')
print()

# Track all harnesses
all_harnesses = []

# Evaluate seeds
print('=== Evaluating seed harnesses ===')
for name, code in SEEDS.items():
    print(f'\n  {name}...')
    scores = evaluate_harness(code, search_problems, retrieval_pool)
    all_harnesses.append({'name': name, 'code': code, 'scores': scores, 'iteration': 0})
    print(f'    acc={scores["accuracy"]:.3f} cost={scores["context_cost"]:.0f} correct={scores.get("correct",0)}/{scores.get("total",0)}')

# Meta-harness iterations: LLM proposes, LLM evaluates
for iteration in range(1, MH_ITERATIONS + 1):
    print(f'\n=== Meta-Harness Iteration {iteration}/{MH_ITERATIONS} ===')
    
    # Build archive summary
    archive_lines = []
    for h in all_harnesses:
        s = h['scores']
        archive_lines.append(f"'{h['name']}': acc={s['accuracy']:.3f}, cost={s['context_cost']:.0f}")
        # Show code preview of best
        if s['accuracy'] == max(x['scores']['accuracy'] for x in all_harnesses):
            archive_lines.append(f"  Code: {h['code'][:300]}...")
    archive_text = '\n'.join(archive_lines)
    
    # Propose
    print('  Proposing...')
    new_code = propose_harness(archive_text, iteration)
    
    # Validate
    try:
        ns = {}
        exec(new_code, ns)
        if not callable(ns.get('run')):
            print('  No run() — skipping'); continue
    except Exception as e:
        print(f'  Invalid code: {e} — skipping'); continue
    
    # Evaluate
    name = f'mh_iter{iteration}'
    print(f'  Evaluating {name}...')
    scores = evaluate_harness(new_code, search_problems, retrieval_pool)
    all_harnesses.append({'name': name, 'code': new_code, 'scores': scores, 'iteration': iteration})
    print(f'    acc={scores["accuracy"]:.3f} cost={scores["context_cost"]:.0f} correct={scores.get("correct",0)}/{scores.get("total",0)}')

# Pick best harness
all_harnesses.sort(key=lambda h: h['scores']['accuracy'], reverse=True)
best = all_harnesses[0]

search_time = time.time() - search_start
print(f'\n{"="*60}')
print(f'Meta-Harness Search complete in {search_time:.0f}s')
print(f'Best: {best["name"]} (acc={best["scores"]["accuracy"]:.3f})')
print(f'All results:')
for h in all_harnesses:
    s = h['scores']
    print(f'  {h["name"]:25s} acc={s["accuracy"]:.3f} cost={s["context_cost"]:.0f}')

# Compile best harness run function
best_ns = {}
exec(best['code'], best_ns)
best_run = best_ns['run']
print(f'\nBest harness loaded: {best["name"]}')

In [ ]:
# ============================================================
# Cell 8: TIR Solver (uses best harness + vLLM server)
# ============================================================

TIR_SYSTEM = '''You are a world-class mathematician. Solve the given problem step by step.
- Write Python code in ```python ... ``` blocks. Output appears in ```output ... ```.
- Use sympy for symbolic math, numpy for numerical work.
- Put your final answer in \\boxed{N} where N is an integer 0-99999.
- Double-check with a verification code block.'''

def tir_solve(problem_text, n_samples=N_SAMPLES):
    """Full TIR pipeline: harness prompt -> generate -> execute code -> vote."""
    # Build prompt using discovered harness
    harness_result = best_run({'question': problem_text, 'corpus': corpus[:500]})
    raw_prompt = harness_result['prompt']
    full_prompt = f"{TIR_SYSTEM}\n\n{raw_prompt}"
    
    # Generate N samples via vLLM server
    responses = vllm_generate(full_prompt, n=n_samples, temperature=TEMPERATURE,
                               max_tokens=MAX_TOKENS, stop=['```output'])
    
    results = []
    for text in responses:
        full_text = text or ''
        code_ok = False
        
        blocks = extract_code_blocks(full_text)
        if blocks:
            so, se, ok = execute_code(blocks[-1])
            code_ok = ok
            so = so or ''
            se = se or ''
            out = so.strip() if ok and so.strip() else (se.strip()[:500] if not ok else '(no output)')
            full_text += f'\n```output\n{out}\n```\n'
            
            # Continue generation after code output
            cont_prompt = full_prompt + full_text
            for _ in range(TIR_MAX_RETRIES):
                try:
                    cont = vllm_generate(cont_prompt, n=1, temperature=0.0,
                                          max_tokens=MAX_TOKENS//2, stop=['```output'])
                    chunk = (cont[0] if cont else '') or ''
                except Exception:
                    break
                if not chunk.strip(): break
                full_text += chunk
                nb = extract_code_blocks(chunk)
                if nb:
                    so, se, ok = execute_code(nb[-1])
                    so = so or ''
                    se = se or ''
                    if ok: code_ok = True
                    out = so.strip() if ok and so.strip() else se.strip()[:500]
                    full_text += f'\n```output\n{out}\n```\n'
                    cont_prompt = full_prompt + full_text
                else: break
                if extract_boxed(full_text) is not None: break
        
        answer = extract_answer(full_text)
        w = 1.0
        if code_ok: w += 2.0
        if '\\boxed' in full_text: w += 0.5
        results.append((answer, w))
    
    # Weighted vote
    wt = {}
    for ans, w in results:
        if ans is not None:
            wt[ans] = wt.get(ans, 0.0) + w
    if not wt:
        return 0, 0.0
    best_ans = max(wt, key=wt.get)
    valid = [a for a, _ in results if a is not None]
    conf = Counter(valid).most_common(1)[0][1] / len(valid) if valid else 0
    return best_ans % ANSWER_MOD, conf

print('TIR solver ready.')

In [ ]:
# ============================================================
# Cell 9: Time Manager + Main Solve
# ============================================================

class Timer:
    def __init__(self, total=32400, per=1700, n=110):
        self.total, self.per, self.n = total, per, n
        self.start = time.time(); self.solved = 0; self.times = []
    def elapsed(self): return time.time() - self.start
    def remaining(self): return max(0, self.total - self.elapsed())
    def budget(self):
        r = max(1, self.n - self.solved)
        return max(60, min(self.remaining()/r, self.per))
    def get_n(self, base=N_SAMPLES):
        b = self.budget()
        if b >= 1500: return base
        elif b >= 900: return max(16, base//2)
        elif b >= 300: return max(8, base//4)
        return 4
    def record(self, t): self.solved += 1; self.times.append(t)
    def skip(self): return self.remaining() < 30
    def status(self):
        a = sum(self.times)/len(self.times) if self.times else 0
        return f'{self.solved}/{self.n}|{self.elapsed():.0f}s|{self.remaining():.0f}s left|avg {a:.1f}s'

timer = Timer()

def solve(problem):
    if timer.skip(): return 0
    t0 = time.time()
    try:
        n = timer.get_n()
        answer, conf = tir_solve(problem, n_samples=n)
        elapsed = time.time() - t0
        timer.record(elapsed)
        print(f'  n={n} conf={conf:.2f} ans={answer} {elapsed:.1f}s | {timer.status()}')
        return answer
    except Exception as e:
        timer.record(time.time()-t0)
        print(f'  ERROR: {e}')
        traceback.print_exc()
        return 0

print('Solver ready.')

In [ ]:
# ============================================================
# Cell 10: Kaggle Submission Server
# ============================================================
import kaggle_evaluation.aimo_3_inference_server

def predict(id_, problem):
    pid = id_.item(0)
    ptxt = problem.item(0)
    print(f'\n{"="*60}\nProblem {timer.solved+1} (id={pid}):')
    print(f'  {ptxt[:120]}...' if len(ptxt) > 120 else f'  {ptxt}')
    gc.disable()
    answer = solve(ptxt)
    gc.enable(); gc.collect()
    answer = int(answer) % ANSWER_MOD
    print(f'  => {answer}')
    return pl.DataFrame({'id': [pid], 'answer': [answer]})

server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    server.serve()
else:
    candidates = (
        glob.glob('/kaggle/input/competitions/*/test.csv') +
        glob.glob('/kaggle/input/*/test.csv') +
        glob.glob('/kaggle/input/*/*/test.csv')
    )
    test_path = candidates[0] if candidates else '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv'
    print(f'Using: {test_path}')
    server.run_local_gateway((test_path,))

print(f'\nDone! {timer.status()}')